## 🛠️ Data Preprocessing

In [ ]:
import json
import pandas as pd
import os
from sklearn.model_selection import StratifiedGroupKFold

In [ ]:
# Loading local dataset configurations (data paths)
with open("config.json") as json_file:
    _LOCAL_CONFIG = json.load(json_file)

DATA_FOLDER_PATH = _LOCAL_CONFIG['dataset_folder_path']
CSV_PATH = os.path.join(DATA_FOLDER_PATH, "anonymous-metadata.csv")
TYPE_CLASS = "diag" # diag or triage
TYPE_IMG = "CLINICAL" # DERMATOSCOPE, CLINICAL or BOTH
TARGET_COLUMN = "macroCIDDiagnostic"
TARGET_NUMBER_COLUMN = "diagnostic-number"
IMG_COLUMN = "img-id"
IMG_SRC_COLUMN = "img-src"

### Data Cleaning and Filtering

In [ ]:
clin_ = ["img-id", "clinicalDiagnostic", "patient-id", "lesion-id", "histoDiagnostic", "macroCIDDiagnostic"]
clin_feats = ["age", "usePesticide", "gender", "familySkinCancerHistory", "familyCancerHistory", 
              "fitzpatrickSkinType", "macroBodyRegion", "hasItched", "hasGrown", "hasHurt", "hasChanged", 
              "hasBled", "hasElevation"]

data_csv = pd.read_csv(CSV_PATH).fillna("EMPTY").replace(" ", "EMPTY").replace("  ", "EMPTY").replace("NÃO  ENCONTRADO", "EMPTY")
data_csv["age"] = data_csv["age"].replace("EMPTY", 0)

if TYPE_IMG != "BOTH":
    data_csv = data_csv[data_csv[IMG_SRC_COLUMN] == TYPE_IMG]

### Class Clustering

In [ ]:
# Selecting only the targets that we want
if TYPE_CLASS == "triage":
    cluster_targets = {
        "C43": "P1",
        "D03": "P1",
        "D22": "P1",

        "C80": "P2",
        "C44": "P2",
        "D04": "P2",
        "L75": "P2",  
        "D23": "P2",    

        "L57": "P3",
        "L25": "P3",
        "L30": "P3",
        "L98": "P3",    

        "L78": "P4",
        "L82": "P4",

        "L70": "P5",
        "00": "P5"
    }
elif TYPE_CLASS == "diag":
    cluster_targets = {
        "C43": "MEL",
        "D03": "MEL",
        "D22": "NEVO",

        "C80": "CBC",
        "C44": "CEC",
        "D04": "CEC",

        "L57": "ACT",    

        "L78": "NEVO",
        "L82": "SEBO",

        # "L70": "ACNE",
        # "00": "PELE"
    }

for idx, row in data_csv.iterrows():

    if row[TARGET_COLUMN] in cluster_targets:
        data_csv.at[idx, TARGET_COLUMN] = cluster_targets[row[TARGET_COLUMN]]
        data_csv.at[idx, IMG_COLUMN] = f"{data_csv.at[idx, IMG_COLUMN]}.png"
    else:        
        data_csv.drop(idx, inplace=True)

print("- Checking the target distribution")
print(data_csv[TARGET_COLUMN].value_counts())
print(f"Total number of samples: {data_csv[TARGET_COLUMN].value_counts().sum()}")

### Metadata Consolidation

In [ ]:
# reset dataframe index
data_csv = data_csv.reset_index(drop=True)

# Selecting the clinical features
cli_feats = dict()
for idx, row in data_csv.iterrows():
    img_id = row[IMG_COLUMN]

    anamnese = ""
    for col in clin_feats:
        anamnese += f"{col}: {row[col]} "    
    
    cli_feats[img_id] = anamnese    

### K-Fold Generation

In [ ]:
# Splitting the dataset
data_train = data_csv[[IMG_COLUMN, TARGET_COLUMN, 'lesion-id']].copy()
kfold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
data_train['folder'] = None
for i, (train_indexes, test_indexes) in enumerate(kfold.split(data_train, data_train[TARGET_COLUMN], groups=data_train['lesion-id'])):
    data_train.loc[test_indexes, 'folder'] = i + 1 

# Converting the labels to numbers
data_train[TARGET_NUMBER_COLUMN] = data_train[TARGET_NUMBER_COLUMN].astype('category')
data_train[TARGET_NUMBER_COLUMN] = data_train[TARGET_NUMBER_COLUMN].cat.codes

data_train.to_csv(os.path.join(DATA_FOLDER_PATH, f"pad-ufes-26-{TYPE_CLASS}_folders_raw.csv"), index=False)

# Saving the data raw
tokens_path = os.path.join(DATA_FOLDER_PATH, "anamnese_raw.json")
with open(tokens_path, 'w') as f:
    json.dump(cli_feats, f, indent=4)
